# Lesson 1.2 — AWS Well-Architected Framework (6 Pillars)

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Name and describe all 6 pillars of the AWS Well-Architected Framework
- Identify design principles and key services for each pillar
- Use the AWS Well-Architected Tool (WA Tool) via CLI to run a workload review
- Evaluate architectural trade-offs using Well-Architected principles

**Exam relevance:** Both SOA-C03 and SAP-C02 — the Well-Architected Framework underpins nearly every exam question.

_Reference: [AWS Well-Architected Framework](https://docs.aws.amazon.com/wellarchitected/latest/framework/welcome.html)_

## The 6 Pillars at a Glance

| # | Pillar | Core Question | Key AWS Services |
|---|--------|--------------|------------------|
| 1 | **Operational Excellence** | How do you run and monitor systems? | CloudFormation, CodePipeline, CloudWatch, Systems Manager |
| 2 | **Security** | How do you protect data and systems? | IAM, KMS, GuardDuty, Security Hub, WAF |
| 3 | **Reliability** | How do you recover from failures? | Route 53, Auto Scaling, ELB, Multi-AZ RDS, AWS Backup |
| 4 | **Performance Efficiency** | How do you use resources efficiently? | EC2 right-sizing, CloudFront, ElastiCache, DynamoDB |
| 5 | **Cost Optimization** | How do you eliminate unnecessary cost? | Cost Explorer, Savings Plans, Spot Instances, Trusted Advisor |
| 6 | **Sustainability** | How do you minimize environmental impact? | Graviton processors, serverless, managed services |

---

## Pillar 1: Operational Excellence

**Goal:** Run workloads effectively, gain operational insights, and continuously improve processes.

### Design Principles
- Perform operations as **code** (IaC, automation, runbooks)
- Make **frequent, small, reversible changes** (CI/CD, blue/green deployments)
- **Anticipate failure** — run game days, chaos engineering
- Learn from **all operational failures**

### CloudOps Engineer Focus Areas
- Automated deployments (CodePipeline, CloudFormation)
- Operational runbooks in Systems Manager
- CloudWatch dashboards and alarms for health visibility

---

## Pillar 2: Security

**Goal:** Protect data, systems, and assets while delivering business value.

### Design Principles
- **Implement a strong identity foundation** — least privilege, IAM roles
- **Enable traceability** — CloudTrail, VPC Flow Logs, Config
- **Apply security at all layers** — network, compute, data
- **Automate security best practices** — AWS Config rules, GuardDuty
- **Protect data in transit and at rest** — TLS, KMS encryption
- **Prepare for security events** — incident response runbooks

---

## Pillar 3: Reliability

**Goal:** Ensure a workload performs its intended function correctly and consistently.

### Design Principles
- **Automatically recover from failure** — health checks, auto scaling
- **Test recovery procedures** — DR drills, chaos engineering
- **Scale horizontally** — distribute load across many small resources
- **Stop guessing capacity** — use auto scaling
- **Manage change through automation** — IaC, change sets

### Key Metrics
- **RTO (Recovery Time Objective):** Max acceptable downtime
- **RPO (Recovery Point Objective):** Max acceptable data loss window

---

## Pillar 4: Performance Efficiency

**Goal:** Use computing resources efficiently to meet system requirements.

### Design Principles
- **Democratize advanced technologies** — use managed services (RDS, DynamoDB) instead of building from scratch
- **Go global in minutes** — multi-region deployment with CloudFront
- **Use serverless architectures** — Lambda, Fargate
- **Experiment more often** — try different instance types
- **Consider mechanical sympathy** — use the right tool for the job

---

## Pillar 5: Cost Optimization

**Goal:** Avoid unnecessary costs.

### Design Principles
- **Implement cloud financial management** — tagging, cost allocation
- **Adopt a consumption model** — pay for what you use
- **Measure overall efficiency** — CloudWatch + Cost Explorer
- **Stop spending money on undifferentiated heavy lifting** — use managed services
- **Analyze and attribute expenditure** — tagging strategy, cost allocation reports

### Purchasing Options (critical for exams)
| Type | Discount | Commitment | Use Case |
|------|----------|-----------|----------|
| On-Demand | 0% | None | Unpredictable workloads |
| Reserved (1yr) | up to 40% | 1 year | Steady-state workloads |
| Reserved (3yr) | up to 60% | 3 years | Very stable workloads |
| Savings Plans | up to 66% | 1–3 years | Flexible commitment |
| Spot | up to 90% | None (interruptible) | Batch, fault-tolerant |

---

## Pillar 6: Sustainability

**Goal:** Minimize environmental impact of AWS workloads.

### Design Principles
- Understand your **environmental impact**
- Establish **sustainability goals**
- **Maximize utilization** — right-size, consolidate
- Adopt **efficient hardware** — Graviton (ARM) processors use 60% less energy
- Use **managed services** — AWS optimizes utilization at scale

In [ ]:
# Lesson 1.2 — Setup
import boto3
import json

# Verify account access
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"User/Role: {identity['Arn']}")
print()
print("Setup complete.")

In [ ]:
# Use the AWS Well-Architected Tool to create and review a workload
# The WA Tool lets you assess your architecture against the 6 pillars

wa = boto3.client('wellarchitected', region_name='us-east-1')

WORKLOAD_NAME = 'CloudOpsLearning-Demo'

try:
    response = wa.create_workload(
        WorkloadName=WORKLOAD_NAME,
        Description='Demo workload for Well-Architected Framework learning',
        Environment='PREPRODUCTION',        # PRODUCTION or PREPRODUCTION
        Regions=['us-east-1'],
        Lenses=['wellarchitected'],          # The WA Framework lens
        ReviewOwner='cloudops-team@example.com',
        Industry='Technology',
        IndustryType='Internet'
    )
    workload_id = response['WorkloadId']
    print(f"Created workload: {WORKLOAD_NAME}")
    print(f"Workload ID: {workload_id}")
except wa.exceptions.ConflictException:
    # Workload already exists — find it
    workloads = wa.list_workloads(WorkloadNamePrefix=WORKLOAD_NAME)
    workload_id = workloads['WorkloadSummaries'][0]['WorkloadId']
    print(f"Workload already exists. ID: {workload_id}")

In [ ]:
# List available lenses in the Well-Architected Tool
# AWS provides specialized lenses for specific domains (serverless, SaaS, IoT, etc.)

wa = boto3.client('wellarchitected', region_name='us-east-1')
lenses = wa.list_lenses(LensStatus='PUBLISHED')

print("Available Well-Architected Lenses:")
for lens in lenses['LensSummaries']:
    print(f"  [{lens['LensAlias']:30}] {lens['LensName']}")

In [ ]:
# Self-Assessment: Map scenarios to Well-Architected pillars
# This mimics how exam questions work — you must identify the relevant pillar

scenarios = [
    {
        "scenario": "Your EC2 instance CPU is consistently at 5% utilization, and you're paying for a large instance.",
        "pillar": "Cost Optimization",
        "action": "Right-size the instance or switch to Spot/Savings Plans"
    },
    {
        "scenario": "A DynamoDB table has no encryption enabled and stores customer PII.",
        "pillar": "Security",
        "action": "Enable DynamoDB encryption at rest using AWS KMS"
    },
    {
        "scenario": "Your application is deployed only in us-east-1. An AZ outage causes 30 min downtime.",
        "pillar": "Reliability",
        "action": "Deploy across multiple AZs using Auto Scaling + ALB"
    },
    {
        "scenario": "Deployments require 2 days of manual steps documented in a wiki.",
        "pillar": "Operational Excellence",
        "action": "Automate deployments with CodePipeline and CloudFormation"
    },
    {
        "scenario": "Database queries take 500ms due to full table scans on a 10M-row table.",
        "pillar": "Performance Efficiency",
        "action": "Add indexes, use read replicas, or implement ElastiCache caching"
    }
]

print("Well-Architected Pillar Mapping Scenarios:\n")
for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['scenario']}")
    print(f"  Pillar : {s['pillar']}")
    print(f"  Action : {s['action']}")
    print()

## Exercise

For each scenario below, identify the **primary Well-Architected pillar** and propose a concrete AWS solution:

In [ ]:
# TODO: For each scenario, fill in the pillar and AWS solution

your_scenarios = [
    {
        "scenario": "S3 buckets have versioning disabled and objects are overwritten accidentally.",
        "pillar": "TODO",
        "solution": "TODO"
    },
    {
        "scenario": "An application serves users in Europe but is deployed only in us-east-1, causing 200ms latency.",
        "pillar": "TODO",
        "solution": "TODO"
    },
    {
        "scenario": "There is no alerting when the production database CPU exceeds 90%.",
        "pillar": "TODO",
        "solution": "TODO"
    }
]

for s in your_scenarios:
    print(f"Scenario: {s['scenario']}")
    print(f"  Your Pillar  : {s['pillar']}")
    print(f"  Your Solution: {s['solution']}")
    print()

In [ ]:
# Cleanup — delete the demo workload from the WA Tool
import boto3

wa = boto3.client('wellarchitected', region_name='us-east-1')

try:
    workloads = wa.list_workloads(WorkloadNamePrefix='CloudOpsLearning-Demo')
    for w in workloads.get('WorkloadSummaries', []):
        wa.delete_workload(WorkloadId=w['WorkloadId'])
        print(f"Deleted workload: {w['WorkloadName']} ({w['WorkloadId']})")
except Exception as e:
    print(f"Cleanup: {e}")

## Summary

The **6 pillars** — Operational Excellence, Security, Reliability, Performance Efficiency, Cost Optimization, and Sustainability — form the lens through which all AWS architectural decisions are evaluated.

## Key Exam Points
- When a question describes a **problem**, always identify which pillar it falls under before selecting an answer
- **Reliability pillar** = RTO/RPO, multi-AZ, DR
- **Cost Optimization** = Savings Plans, Reserved Instances, right-sizing, Spot
- **Security** = encryption, IAM, least privilege, GuardDuty, Config
- The WA Tool can be used to formally review and track architectural improvements

---

**Next:** [Lesson 1.3 — AWS CLI & boto3 Automation Fundamentals](lesson-1.3-aws-cli-boto3.ipynb)